In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import col, coalesce, lit, current_timestamp
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "proyecto_olist")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_products = spark.table(f"{catalogo}.{esquema_source}.products")

In [0]:
df_products_clean = (
    df_products
    .select(
        col("product_id"),
        col("product_category_name"),
        col("product_name_lenght"),
        col("product_description_lenght"),
        col("product_photos_qty"),
        col("product_weight_g"),
        col("product_length_cm"),
        col("product_height_cm"),
        col("product_width_cm")
    )
    .withColumn(
        "product_category_name",
        coalesce(col("product_category_name"), lit("sin_categoria"))
    )
    .withColumn("silver_insert_date", current_timestamp())
)

In [0]:
df_products_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.products")

In [0]:
print("Tabla silver.products procesada.")